# 🔬 APA (Adaptive Precision Ascension) — Master Research & Forensic Analyzer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RedSafir/Adaptive-Precision-Ascension/blob/main/analyze_forensic.ipynb)

Notebook ini adalah **analisis komprehensif berstandar publikasi ilmiah** untuk framework **Adaptive Precision Ascension (APA)**.

Notebook ini mencakup **5 Pilar Analisis Utama**:
1. **Per-Layer Time Series (Dual-Axis / Stacked Subplot)**:
   - Log-scale $A_{\max}(t)$ vs $T_{\max}$ (ambang batas overflow).
   - Underflow ratio $\bar{\rho}(t)$ vs $\theta = 0.40$ (ambang batas underflow).
   - Subplot bertumpuk untuk melihat apakah eskalasi dipicu oleh ledakan sinyal (*overflow*) atau lenyapnya sinyal (*underflow*).
2. **Pola Lintas-Layer (Arsitektural)**:
   - **Heatmap Depth $\times$ Step**: Sumbu Y = urutan layer (input ke output), Sumbu X = step, Warna = presisi (FP8/FP16/TF32).
   - **Stacked Area Chart Precision Drift**: Persentase layer di FP8 vs FP16 vs TF32 dari waktu ke waktu.
   - **Time-to-Escalation per Layer**: Berapa lama tiap layer bertahan di FP8 sebelum eskalasi pertama.
3. **Root-Cause / Tensor-Role Analysis**:
   - Distribusi agregat `culprit_tensor_role` (apakah forward atau backward yang lebih rentan?).
   - Sebaran nilai elemen tensor (`mean` vs `std`) saat eskalasi terjadi.
4. **Konsekuensi terhadap Training Health**:
   - Overlay garis vertikal eskalasi di atas kurva training loss.
   - Kurva kumulatif batch yang di-skip (*early-crawl phase* validation).
   - Kuantifikasi degradasi throughput akibat precision drift.
5. **Sensitivitas Hyperparameter / Ablasi**:
   - Komparasi multi-run dengan variasi $\gamma$ dan $\theta_{\text{underflow}}$.

## 1. Setup Environment & Ingesti Data (Google Drive / Local)
Sel ini menghubungkan Google Colab ke Google Drive (`/content/drive/MyDrive/result/`) dan memuat semua library visualisasi.

In [ ]:
import os
import json
import glob
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import networkx as nx
from collections import Counter, defaultdict

# Setup style visualisasi paper-ready
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'

# Deteksi Google Colab & Auto-Mount Google Drive
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("💻 Terdeteksi di Google Colab. Menghubungkan ke Google Drive...")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Google Drive berhasil di-mount di /content/drive!")
    except Exception as e:
        print(f"ℹ️ Google Drive mount info: {e}")

# Pola pencarian file log forensik & reguler
search_patterns = [
    '/content/drive/MyDrive/result/*forensic*.jsonl',
    '/content/drive/My Drive/result/*forensic*.jsonl',
    '/content/drive/MyDrive/result/*.jsonl',
    '/content/drive/My Drive/result/*.jsonl',
    '/content/drive/MyDrive/*forensic*.jsonl',
    'result/*forensic*.jsonl',
    'result/*.jsonl',
    '*forensic*.jsonl',
    '*.jsonl'
]

all_detected_files = []
for pat in search_patterns:
    all_detected_files.extend(glob.glob(pat))

all_detected_files = list(dict.fromkeys(all_detected_files))
print(f"📁 Total file log terdeteksi: {len(all_detected_files)}")
for idx, f in enumerate(all_detected_files):
    size_kb = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f"  [{idx:2d}] {f:<45s} ({size_kb:6.1f} KB)")

# Pilih file forensik aktif & file log reguler aktif
forensic_files = [f for f in all_detected_files if 'forensic' in f.lower()]
regular_files = [f for f in all_detected_files if 'forensic' not in f.lower()]

ACTIVE_FORENSIC_FILE = forensic_files[0] if forensic_files else (all_detected_files[0] if all_detected_files else None)
ACTIVE_REGULAR_FILE = regular_files[0] if regular_files else None

print(f"\n👉 File Forensik Utama  : {ACTIVE_FORENSIC_FILE}")
print(f"👉 File Training Utama  : {ACTIVE_REGULAR_FILE}")

## 2. Multi-Source Log Ingestion & Data Unification
Mem-parsing semua data dari file log forensik dan file training reguler ke dalam DataFrame yang terstruktur rapi.

In [ ]:
def parse_comprehensive_logs(forensic_path, regular_path=None):
    escalation_records = []
    role_stats_records = []
    periodic_telemetry_records = []
    skip_records = []
    epoch_records = []
    step_loss_records = []

    files_to_read = [f for f in [forensic_path, regular_path] if f and os.path.exists(f)]

    for filepath in files_to_read:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                except json.JSONDecodeError:
                    continue

                event_type = data.get('event')
                
                # 1. Escalation Event
                if event_type == 'escalation' or 'culprit_tensor_role' in data:
                    shape = data.get('tensor_shape')
                    shape_str = ' x '.join(map(str, shape)) if isinstance(shape, list) else str(shape)
                    per_role = data.get('per_role_amax') or {}
                    
                    esc_dict = {
                        'step': data.get('step', 0),
                        'module_name': data.get('module_name') or data.get('module'),
                        'reason': data.get('reason'),
                        'level_before': data.get('level_before', data.get('old_level')),
                        'level_after': data.get('level_after', data.get('new_level')),
                        'culprit_role': data.get('culprit_tensor_role'),
                        'trigger_value': data.get('amax_value', data.get('trigger_value')),
                        'threshold': data.get('threshold_at_time', 403.2),
                        'tensor_shape': shape_str,
                        'dtype': data.get('dtype_at_time'),
                        'preceding_module': data.get('preceding_module_in_forward_order', '(None)'),
                        'underflow_ratio': data.get('underflow_ratio'),
                        'timestamp_utc': data.get('timestamp_utc', data.get('timestamp')),
                        'amax_input': per_role.get('input_activation'),
                        'amax_weight': per_role.get('weight'),
                        'amax_output': per_role.get('output'),
                        'amax_grad_out': per_role.get('grad_output'),
                        'amax_grad_weight': per_role.get('grad_weight'),
                        'amax_grad_in': per_role.get('grad_input'),
                    }
                    escalation_records.append(esc_dict)

                    # Per-role stats
                    per_stats = data.get('per_role_stats') or {}
                    if isinstance(per_stats, dict):
                        for role, st in per_stats.items():
                            if isinstance(st, dict):
                                role_stats_records.append({
                                    'step': data.get('step', 0),
                                    'module_name': esc_dict['module_name'],
                                    'role': role,
                                    'mean': st.get('mean'),
                                    'std': st.get('std'),
                                    'amax': per_role.get(role)
                                })

                # 2. Skip Batch Event
                elif event_type == 'skip_batch':
                    skip_records.append({
                        'step': data.get('step', 0),
                        'reason': data.get('reason'),
                        'trigger_modules': data.get('trigger_modules', [])
                    })

                # 3. Periodic Telemetry Record
                elif event_type == 'periodic_telemetry':
                    st = data.get('step', 0)
                    telem = data.get('telemetry', {})
                    for mod, mdata in telem.items():
                        periodic_telemetry_records.append({
                            'step': st,
                            'module_name': mod,
                            'amax': mdata.get('amax'),
                            'underflow_ratio': mdata.get('underflow_ratio'),
                            'ema_underflow': mdata.get('ema_underflow'),
                            'level': mdata.get('level'),
                            'threshold_max': mdata.get('threshold_max')
                        })

                # 4. Epoch Summary
                elif 'epoch' in data and 'train_loss' in data:
                    epoch_records.append(data)

                # 5. Step Loss
                elif 'step' in data and 'loss' in data:
                    step_loss_records.append(data)

    df_esc = pd.DataFrame(escalation_records)
    df_stats = pd.DataFrame(role_stats_records)
    df_telem = pd.DataFrame(periodic_telemetry_records)
    df_skips = pd.DataFrame(skip_records)
    df_epochs = pd.DataFrame(epoch_records)
    df_losses = pd.DataFrame(step_loss_records)
    
    return df_esc, df_stats, df_telem, df_skips, df_epochs, df_losses

df_esc, df_stats, df_telem, df_skips, df_epochs, df_losses = parse_comprehensive_logs(ACTIVE_FORENSIC_FILE, ACTIVE_REGULAR_FILE)

print(f"✅ Berhasil mem-parsing:")
print(f"  • Escalation Events       : {len(df_esc)}")
print(f"  • Role Stats Snapshots    : {len(df_stats)}")
print(f"  • Periodic Telemetry Rows : {len(df_telem)}")
print(f"  • Skipped Batches         : {len(df_skips)}")
print(f"  • Epoch Summaries         : {len(df_epochs)}")

## 3. PILAR 1: Per-Layer Time Series (Dual-Axis Subplot per Layer)
> **Poin Kunci:**
> 1. **Panel Atas ($A_{\max}$ vs $T_{\max}$, Skala Log)**: Memperlihatkan magnitudo aktivasi/bobot terhadap batas overflow ($T_{\max}$). Skala log krusial karena rentang FP8 (448) vs TF32 ($3.4 \times 10^{38}$) terlampau jauh.
> 2. **Panel Bawah (Underflow Ratio $\bar{\rho}$ vs $\theta = 0.40$)**: Memperlihatkan fraksi gradien yang sekarat (< 0.015625).
> 3. **Penanda Eskalasi Vertikal**: Garis putus-putus merah/oranye di kedua panel yang menandai waktu eskalasi terjadi.

In [ ]:
# Tentukan daftar layer yang ingin di-plot (mis. layer yang mengalami eskalasi)
if not df_esc.empty:
    sample_modules = list(df_esc['module_name'].unique()[:4])
elif not df_telem.empty:
    sample_modules = list(df_telem['module_name'].unique()[:4])
else:
    sample_modules = []

if sample_modules:
    for mod_name in sample_modules:
        fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(15, 6), sharex=True, gridspec_kw={'height_ratios': [1.2, 1]})
        
        # Ambil data eskalasi untuk modul ini
        mod_esc = df_esc[df_esc['module_name'] == mod_name]
        
        # Panel Atas: Amax vs Tmax (Log Scale)
        if not df_telem.empty and mod_name in df_telem['module_name'].values:
            sub_t = df_telem[df_telem['module_name'] == mod_name].sort_values('step')
            ax_top.plot(sub_t['step'], sub_t['amax'], label='Layer $A_{max}$', color='#2563EB', linewidth=2, marker='.')
            ax_top.step(sub_t['step'], sub_t['threshold_max'], label='Threshold $T_{max}$', color='#DC2626', linestyle='--', where='post', alpha=0.8)
        else:
            # Gunakan data dari event eskalasi jika telemetri periodik belum ada
            if not mod_esc.empty:
                ax_top.scatter(mod_esc['step'], mod_esc['trigger_value'], color='#2563EB', s=80, label='Amax @ Event', zorder=5)
                ax_top.step(mod_esc['step'], mod_esc['threshold'], label='Threshold $T_{max}$', color='#DC2626', linestyle='--', where='post')
        
        ax_top.set_yscale('log')
        ax_top.set_ylabel('$A_{max}$ (Log Scale)', fontweight='bold')
        ax_top.set_title(f'Layer Telemetry Dynamics: {mod_name}', fontsize=13, fontweight='bold')
        ax_top.legend(loc='upper left')
        ax_top.grid(True, which="both", ls="--", alpha=0.3)
        
        # Panel Bawah: Underflow Ratio (rho) vs Theta
        if not df_telem.empty and mod_name in df_telem['module_name'].values:
            sub_t = df_telem[df_telem['module_name'] == mod_name].sort_values('step')
            ax_bot.plot(sub_t['step'], sub_t['ema_underflow'], label='EMA Underflow Ratio ($\bar{\rho}$)', color='#F59E0B', linewidth=2)
        elif not mod_esc.empty:
            underflow_pts = mod_esc[mod_esc['reason'] == 'SILENT_UNDERFLOW']
            if not underflow_pts.empty:
                ax_bot.scatter(underflow_pts['step'], underflow_pts['trigger_value'], color='#F59E0B', s=80, label='$\bar{\rho}$ @ Event', zorder=5)
        
        ax_bot.axhline(0.40, color='#B91C1C', linestyle=':', linewidth=2, label='Threshold $\theta = 0.40$')
        ax_bot.set_ylim(-0.05, 1.05)
        ax_bot.set_xlabel('Training Step', fontweight='bold')
        ax_bot.set_ylabel('Underflow Ratio $\bar{\rho}$', fontweight='bold')
        ax_bot.legend(loc='upper left')
        ax_bot.grid(True, alpha=0.3)
        
        # Mark event eskalasi dengan garis vertikal di kedua subplot
        if not mod_esc.empty:
            for _, row in mod_esc.iterrows():
                ev_color = '#DC2626' if row['reason'] == 'OVERFLOW' else '#F59E0B'
                ax_top.axvline(row['step'], color=ev_color, linestyle='--', alpha=0.7)
                ax_bot.axvline(row['step'], color=ev_color, linestyle='--', alpha=0.7)
                trans_label = f"{row['reason']}\n({row['level_before']}➜{row['level_after']})"
                ax_top.text(row['step'], ax_top.get_ylim()[0]*5, trans_label, color=ev_color, fontsize=9, fontweight='bold', ha='right', va='bottom', rotation=90)
        
        plt.tight_layout()
        plt.show()
else:
    print("ℹ️ Belum ada modul yang tercatat mengalami eskalasi.")

## 4. PILAR 2: Pola Lintas-Layer (Arsitektural Heatmap & Drift)
> **Poin Kunci:**
> 1. **Heatmap Depth $\times$ Step**: Sumbu Y = urutan kedalaman layer, Sumbu X = step, Warna = presisi (Hijau=FP8, Biru=FP16, Merah=TF32).
> 2. **Stacked Area Chart (Precision Drift)**: Memvisualisasikan penyusutan area FP8 seiring training berjalan.
> 3. **Time-to-Escalation**: Menguji apakah kegagalan terkonsentrasi di layer tertentu (misal Attention out_proj vs MLP).

In [ ]:
if not df_esc.empty or not df_epochs.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 6))

    # 1. Stacked Area Chart: Precision Drift Over Epochs / Steps
    if not df_epochs.empty and 'precision_distribution' in df_epochs.columns:
        dist_data = []
        for _, r in df_epochs.iterrows():
            pd_dict = r['precision_distribution']
            if isinstance(pd_dict, dict):
                tot = pd_dict.get('fp8', 0) + pd_dict.get('fp16', 0) + pd_dict.get('tf32', 0)
                if tot > 0:
                    dist_data.append({
                        'epoch': r['epoch'],
                        'FP8': (pd_dict.get('fp8', 0) / tot) * 100,
                        'FP16': (pd_dict.get('fp16', 0) / tot) * 100,
                        'TF32': (pd_dict.get('tf32', 0) / tot) * 100
                    })
        if dist_data:
            df_drift = pd.DataFrame(dist_data)
            ax1.stackplot(df_drift['epoch'], df_drift['FP8'], df_drift['FP16'], df_drift['TF32'],
                          labels=['FP8 (Speedup)', 'FP16 (Medium)', 'TF32 (Extended)'],
                          colors=['#10B981', '#3B82F6', '#EF4444'], alpha=0.85)
            ax1.set_xlabel('Epoch')
            ax1.set_ylabel('Proporsi Presisi Jaringan (%)')
            ax1.set_ylim(0, 100)
            ax1.set_title('Global Precision Drift (Escalation Dynamics)')
            ax1.legend(loc='upper right')
            ax1.grid(True, alpha=0.3)
    else:
        ax1.text(0.5, 0.5, 'Data Precision Distribution per-epoch belum tersedia', ha='center', va='center')

    # 2. Time-to-Escalation per Layer (Bar Chart)
    if not df_esc.empty:
        first_esc = df_esc.groupby('module_name')['step'].min().sort_values()
        y_pos = np.arange(len(first_esc))
        ax2.barh(y_pos, first_esc.values, color='#6366F1', edgecolor='black', alpha=0.85)
        ax2.set_yticks(y_pos)
        ax2.set_yticklabels(first_esc.index, fontsize=10)
        ax2.set_xlabel('Training Step Bertahan di FP8 Sebelum Eskalasi')
        ax2.set_title('Time-to-Escalation per Layer (Ketahanan di FP8)')
        for i, val in enumerate(first_esc.values):
            ax2.text(val + 1, i, f'Step {val}', va='center', fontsize=9, fontweight='bold')
        ax2.grid(True, alpha=0.3)
        ax2.invert_yaxis()

    plt.tight_layout()
    plt.show()

    # 3. Heatmap Depth x Step
    if not df_esc.empty:
        unique_mods = list(df_esc['module_name'].unique())
        max_step = df_esc['step'].max()
        step_bins = np.linspace(0, max_step, min(30, max(5, int(max_step/50))))
        
        # Bangun matriks presisi (modul x step)
        # Inisialisasi semua di Level 0 (FP8)
        mat = np.zeros((len(unique_mods), len(step_bins)))
        for m_idx, m_name in enumerate(unique_mods):
            m_events = df_esc[df_esc['module_name'] == m_name].sort_values('step')
            cur_lvl = 0
            for s_idx, s_val in enumerate(step_bins):
                passed_events = m_events[m_events['step'] <= s_val]
                if not passed_events.empty:
                    last_ev = passed_events.iloc[-1]
                    lvl_str = str(last_ev['level_after'])
                    cur_lvl = 2 if 'TF32' in lvl_str or lvl_str == '2' else (1 if 'FP16' in lvl_str or lvl_str == '1' else 0)
                mat[m_idx, s_idx] = cur_lvl
                
        plt.figure(figsize=(14, max(5, len(unique_mods) * 0.45)))
        from matplotlib.colors import ListedColormap
        cmap = ListedColormap(['#10B981', '#3B82F6', '#EF4444'])
        
        sns.heatmap(mat, yticklabels=unique_mods, xticklabels=[f'{int(s)}' for s in step_bins], cmap=cmap, cbar=False, linewidths=0.5, linecolor='white')
        plt.title('Arsitektural Heatmap: Depth x Step Precision Level', fontsize=13, fontweight='bold')
        plt.xlabel('Training Step')
        plt.ylabel('Module (Architectural Depth)')
        
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor='#10B981', label='FP8 (Level 0)'),
            Patch(facecolor='#3B82F6', label='FP16 (Level 1)'),
            Patch(facecolor='#EF4444', label='TF32 (Level 2)')
        ]
        plt.legend(handles=legend_elements, loc='upper right', framealpha=0.9)
        plt.tight_layout()
        plt.show()

## 5. PILAR 3: Root-Cause & Tensor-Role Analysis (Data Forensik)
> **Poin Kunci:**
> 1. **Distribusi Culprit Role Agregat**: Berapa persen kegagalan dipicu oleh `input_activation`, `weight`, `output`, `grad_output`, `grad_weight`, atau `grad_input`?
> 2. **Forward vs Backward Failure**: Apakah instabilitas lebih sering terjadi di fase forward pass atau backward pass?
> 3. **Distribusi Nilai Elemen Tensor**: Boxplot log-scale variasi `std` dan `mean` untuk membedakan outlier ekstrem vs distorsi distribusi menyeluruh.

In [ ]:
if not df_esc.empty and 'culprit_role' in df_esc.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

    # 1. Agregat Culprit Role Bar Chart
    role_counts = df_esc['culprit_role'].value_counts(dropna=False)
    role_colors = {
        'input_activation': '#3B82F6', 
        'output': '#EF4444', 
        'weight': '#10B981', 
        'grad_output': '#8B5CF6',
        'grad_weight': '#EC4899',
        'grad_input': '#F59E0B'
    }
    bar_c = [role_colors.get(r, '#6B7280') for r in role_counts.index]
    
    sns.barplot(x=role_counts.values, y=role_counts.index.astype(str), palette=bar_c, ax=ax1)
    ax1.set_xlabel('Jumlah Kejadian Sebagai Culprit')
    ax1.set_ylabel('Peran Tensor (Tensor Role)')
    ax1.set_title('Distribusi Biang Kerok Eskalasi (Culprit Role)')
    for i, v in enumerate(role_counts.values):
        pct = (v / len(df_esc)) * 100
        ax1.text(v + 0.1, i, f'{v} ({pct:.1f}%)', va='center', fontweight='bold')

    # 2. Forward vs Backward Phase Breakdown
    def classify_phase(role):
        if pd.isna(role):
            return 'Unknown'
        return 'Backward Phase' if 'grad' in str(role).lower() else 'Forward Phase'

    phase_series = df_esc['culprit_role'].apply(classify_phase)
    phase_counts = phase_series.value_counts()
    ax2.pie(phase_counts, labels=phase_counts.index, autopct='%1.1f%%', colors=['#3B82F6', '#EC4899', '#9CA3AF'], startangle=140, wedgeprops=dict(width=0.45))
    ax2.set_title('Fase Terjadinya Instabilitas (Forward vs Backward)')

    plt.tight_layout()
    plt.show()
    
    # 3. Boxplot Log-scale Nilai Std dan Amax per Role (Distribusi Outlier vs Bulk)
    if not df_stats.empty:
        plt.figure(figsize=(12, 5))
        df_stats['std_safe'] = df_stats['std'].apply(lambda x: max(x, 1e-12) if pd.notnull(x) else np.nan)
        sns.boxplot(data=df_stats, x='role', y='std_safe', palette='Set2')
        plt.yscale('log')
        plt.xticks(rotation=20, ha='right')
        plt.xlabel('Peran Tensor (Role)')
        plt.ylabel('Standar Deviasi (Log Scale)')
        plt.title('Dispersi Nilai Elemen Tensor (Std) Saat Insiden Terjadi', fontsize=13, fontweight='bold')
        plt.grid(True, which="both", ls="--", alpha=0.3)
        plt.tight_layout()
        plt.show()

## 6. PILAR 4: Konsekuensi terhadap Training Health
> **Poin Kunci:**
> 1. **Overlay Escalation ke Training Loss**: Memverifikasi apakah eskalasi menyebabkan lonjakan loss atau transisi berlangsung mulus tanpa merusak konvergensi.
> 2. **Cumulative Batch Skipped**: Validasi empiris fenomena *early-crawl phase* (batch yang dibuang untuk melindungi master weights).
> 3. **Throughput Tracking**: Kuantifikasi penurunan kecepatan training saat precision drift terjadi.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 5))

# 1. Training Loss Curve with Escalation Markers
if not df_epochs.empty and 'train_loss' in df_epochs.columns:
    ax1.plot(df_epochs['epoch'], df_epochs['train_loss'], label='Train Loss', color='#2563EB', linewidth=2.5)
    if 'test_loss' in df_epochs.columns:
        ax1.plot(df_epochs['epoch'], df_epochs['test_loss'], label='Test Loss', color='#DC2626', linewidth=2, linestyle='--')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training Loss Convergence & Escalation Overlay')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
elif not df_losses.empty:
    ax1.plot(df_losses['step'], df_losses['loss'], label='Step Loss', color='#2563EB', linewidth=1.5)
    # Overlay marker eskalasi di step terkait
    if not df_esc.empty:
        for _, r in df_esc.iterrows():
            ax1.axvline(r['step'], color='#EF4444', linestyle=':', alpha=0.6)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.set_title('Step Loss Curve with Escalation Events')
    ax1.grid(True, alpha=0.3)

# 2. Cumulative Batch Skipped Over Time
if not df_skips.empty:
    df_skips['cum_skips'] = np.arange(1, len(df_skips) + 1)
    ax2.step(df_skips['step'], df_skips['cum_skips'], color='#EF4444', linewidth=2.5, where='post', label='Kumulatif Batch Skip')
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Total Batch Dibuang (Skip)')
    ax2.set_title('Kumulatif Batch Skipped (Early-Crawl Validation)')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, '✅ 0 Batch Skipped! (Tidak ada hard overflow / NaN)', ha='center', va='center', fontsize=12, fontweight='bold', color='#10B981')
    ax2.set_title('Kumulatif Batch Skipped')

plt.tight_layout()
plt.show()

# 3. Analisis Throughput (Waktu per Epoch)
if not df_epochs.empty and 'epoch_time_sec' in df_epochs.columns:
    plt.figure(figsize=(10, 4))
    plt.plot(df_epochs['epoch'], df_epochs['epoch_time_sec'], marker='o', color='#8B5CF6', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Waktu Eksekusi (Detik / Epoch)')
    plt.title('Throughput Waktu per Epoch (Dampak Precision Drift terhadap Kecepatan)')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. PILAR 5: Sensitivitas Hyperparameter / Ablasi (Multi-Log Comparison)
Membandingkan beberapa eksperimen dengan konfigurasi hyperparameter $\gamma$ (safety margin) dan $\theta_{\text{underflow}}$ berbeda untuk mengukur trade-off antara kestabilan dan efisiensi.

In [ ]:
# Bandingkan semua file log yang terdeteksi di folder
comparison_rows = []

for f in all_detected_files:
    h_info = {}
    e_count = 0
    skip_cnt = 0
    final_loss = '-'
    final_acc = '-'
    avg_time = '-'
    
    try:
        with open(f, 'r', encoding='utf-8') as fl:
            for line in fl:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                if not h_info and ('config' in d or 'args' in d):
                    h_info = d
                if d.get('event') == 'escalation' or 'culprit_tensor_role' in d:
                    e_count += 1
                elif d.get('event') == 'skip_batch':
                    skip_cnt += 1
                elif 'epoch' in d and 'train_loss' in d:
                    final_loss = f"{d['train_loss']:.4f}"
                    final_acc = f"{d.get('test_acc', 0)*100:.2f}%"
                    avg_time = f"{d.get('epoch_time_sec', 0):.2f}s"
    except Exception:
        continue
        
    gamma = h_info.get('config', {}).get('gamma', h_info.get('args', {}).get('gamma', '-'))
    theta = h_info.get('config', {}).get('theta_underflow', h_info.get('args', {}).get('theta_underflow', '-'))
    preset = h_info.get('args', {}).get('apa_preset', '-')
    
    comparison_rows.append({
        'Log File': os.path.basename(f),
        'Preset': preset,
        'Gamma (γ)': gamma,
        'Theta (θ)': theta,
        'Total Escalations': e_count,
        'Total Skips': skip_cnt,
        'Final Loss': final_loss,
        'Final Acc': final_acc,
        'Epoch Time': avg_time
    })

df_ablation = pd.DataFrame(comparison_rows)
print("🏆 Tabel Hasil Komparasi Sensitivitas & Ablasi:")
display(df_ablation)

## 8. Ekspor Ringkasan Laporan ke CSV
Sel ini menyimpan seluruh tabel event eskalasi dan metrik ke file CSV yang siap diunduh ke komputer Anda.

In [ ]:
if not df_esc.empty:
    output_csv = 'apa_master_research_summary.csv'
    df_esc.to_csv(output_csv, index=False)
    print(f"✅ Berhasil mengekspor ringkasan forensik riset ke: {output_csv}")
    
    if IN_COLAB:
        from google.colab import files
        files.download(output_csv)
else:
    print("ℹ️ Data kosong.")